# 34-量化策略全流程（面试级项目）

> 终极项目 | 多因子 ML 选股策略 | 从研究思路到完整研报

## 项目定位

这是 0-100 Quant 的收官项目。目标是把前面 33 节课的所有知识串成一条完整的量化研究流水线，产出一份可以在面试中展示的作品。

## 学习目标

- 独立完成一个量化策略从 idea 到研报的全流程
- 掌握面试中如何讲清楚"你做的策略为什么有效"
- 证明你能严格避免回测陷阱、含仓位管理/止损、有归因分析

## 策略方向：多因子 ML 选股

选择这个方向是因为它整合了课程中最核心的知识点：因子构建（29）+ ML 建模（30）+ 回测引擎（32）+ 风控（33）+ 归因（27）。

## 环境依赖

`numpy`、`pandas`、`matplotlib`。全部使用模拟数据 + 固定随机种子，可在任何环境复现。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Tuple
from enum import Enum
import warnings

warnings.filterwarnings("ignore")
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("=" * 60)
print("  0-100 Quant 终极项目：多因子 ML 选股策略")
print("  复现种子: 42 | 所有随机结果可完全复现")
print("=" * 60)

## 第一阶段：策略逻辑

### 投资哲学

> 便宜的好公司，在市场情绪悲观时买入，在情绪恢复时卖出。

这不是数据挖掘——每个因子都有经济学故事：

| 因子类别 | 因子 | 经济学逻辑 | 来源 |
|---------|------|-----------|------|
| 估值 | 价格/均线偏离 | 均值回归：偏离过大时有回复动力 | 行为金融 |
| 动量 | 短期收益 | 趋势延续：强者短期继续强 | Jegadeesh-Titman |
| 低波动 | 波动率 | 低波动异象：低风险股票收益反而更高 | Ang et al. |
| 质量 | 收益率稳定性 | 盈利稳定的公司风险溢价更低 | Fama-French |

### 策略框架

```text
多只股票 × 多个因子 → ML 打分 → 选 top K 只 → 等权/凯利配置 → 月度调仓
```

### 为什么不用日频？

- 日频信号噪声太大，手续费会吃掉超额
- 月频调仓交易成本可控，信号也更稳定
- 面试中月频策略更容易讲清楚逻辑

## 第二阶段：数据准备

模拟 10 只股票、5 年的月度数据。每只股票有不同的特性（高动量/低波动/均值回归等），让因子有区分度。

In [ ]:
def generate_multi_stock_data(n_stocks=10, n_months=60):
    """生成多股票月度数据

    每只股票有不同的特性参数：
    - 趋势强度（momentum）
    - 波动率水平
    - 均值回归速度
    """
    np.random.seed(RANDOM_SEED)
    dates = pd.date_range("2021-01-01", periods=n_months, freq="ME")

    stocks_data = {}
    for s in range(n_stocks):
        # 每只股票的个性化参数
        stock_seed = RANDOM_SEED + s * 100
        rng = np.random.RandomState(stock_seed)

        trend = rng.uniform(-0.005, 0.015)  # 月度趋势
        vol = rng.uniform(0.04, 0.12)        # 月度波动率
        mr_speed = rng.uniform(0.05, 0.3)    # 均值回归速度

        # 生成价格（含均值回归成分）
        log_price = np.zeros(n_months)
        log_price[0] = np.log(100 + rng.uniform(-20, 20))
        for t in range(1, n_months):
            # 趋势 + 均值回归 + 噪声
            mr = -mr_speed * (log_price[t-1] - 4.6)  # 回归到 ~100
            noise = rng.randn() * vol
            log_price[t] = log_price[t-1] + trend + mr + noise

        price = np.exp(log_price)
        returns = np.diff(log_price, prepend=0)

        df = pd.DataFrame({
            "price": price,
            "return": returns,
            "trend": trend,
            "vol": vol,
            "mr_speed": mr_speed,
        }, index=dates)
        stocks_data[f"stock_{s}"] = df

    return stocks_data, dates

stocks_data, dates = generate_multi_stock_data(n_stocks=10, n_months=60)

# 展示前 3 只股票
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i in range(3):
    df = stocks_data[f"stock_{i}"]
    axes[i].plot(df.index, df["price"], linewidth=1)
    axes[i].set_title(f"Stock {i} | trend={df['trend'].iloc[0]:.3f} vol={df['vol'].iloc[0]:.2f}")
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print(f"模拟股票数: {len(stocks_data)}")
print(f"月度数: {len(dates)}")
print(f"日期范围: {dates[0].date()} ~ {dates[-1].date()}")

## 第三阶段：因子构建

对每只股票每月计算 6 个因子。这是整个策略的"原材料"——因子质量决定了策略上限。

In [ ]:
def build_factors(stocks_data, dates, lookback=12):
    """为所有股票构建月度因子

    Returns:
        DataFrame: index=[date, stock], columns=因子
    """
    records = []

    for sym, df in stocks_data.items():
        price = df["price"].values
        ret = df["return"].values

        for t in range(lookback, len(price)):
            window_price = price[t-lookback:t+1]
            window_ret = ret[t-lookback:t+1]

            # 动量因子
            mom_1m = ret[t]                         # 最近1月收益
            mom_3m = np.sum(window_ret[-3:])         # 最近3月累计
            mom_12m = np.sum(window_ret[:-1]) if len(window_ret) >= 12 else np.sum(window_ret)

            # 波动率因子（低波动异象）
            vol_3m = np.std(window_ret[-3:]) if len(window_ret) >= 3 else np.std(window_ret)
            vol_12m = np.std(window_ret)

            # 估值因子（价格/均线偏离）
            ma_12m = np.mean(window_price)
            ma_dev = (price[t] - ma_12m) / ma_12m  # 正值=超买，负值=超卖

            # 质量因子（收益率稳定性 = 1/变异系数）
            mean_ret = np.mean(window_ret)
            std_ret = np.std(window_ret)
            quality = mean_ret / (std_ret + 1e-8)  # Sharpe-like

            records.append({
                "date": dates[t],
                "stock": sym,
                "mom_1m": mom_1m,
                "mom_3m": mom_3m,
                "mom_12m": mom_12m,
                "vol_3m": -vol_3m,        # 取负：低波动→高分
                "vol_12m": -vol_12m,
                "ma_dev": -ma_dev,         # 取负：超卖→高分（均值回归）
                "quality": quality,
            })

    factor_df = pd.DataFrame(records)
    factor_df.set_index(["date", "stock"], inplace=True)
    return factor_df

factor_df = build_factors(stocks_data, dates, lookback=12)
factor_cols = ["mom_1m", "mom_3m", "mom_12m", "vol_3m", "vol_12m", "ma_dev", "quality"]

print(f"因子数据: {len(factor_df)} 行 (日期×股票)")
print(f"因子列表: {factor_cols}")
print(f"\n因子统计:")
print(factor_df[factor_cols].describe().round(4).to_string())

# 因子相关性
corr = factor_df[factor_cols].corr()
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(factor_cols)))
ax.set_xticklabels(factor_cols, rotation=45, ha="right")
ax.set_yticks(range(len(factor_cols)))
ax.set_yticklabels(factor_cols)
plt.colorbar(im, ax=ax)
ax.set_title("因子相关性矩阵")
plt.tight_layout()
plt.show()

## 第四阶段：模型训练

使用线性模型做因子合成（等权 → 回归权重 → IC 加权），对比三种方法的样本外表现。

核心原则：
- 时间顺序切分（前 36 个月训练，后 24 个月测试）
- 所有标准化参数只从训练集计算
- 标签是下个月的收益

In [ ]:
# 构造标签（下月收益）+ 时间切分
factor_df["next_return"] = factor_df.groupby("stock")["mom_1m"].shift(-1)  # 用 mom_1m 定位原始收益

# 更准确的标签：用价格算下月收益
def build_labels_and_split(factor_df, stocks_data, dates, train_months=36):
    """构造标签并切分训练/测试集"""
    labels = []
    for (d, sym), row in factor_df.iterrows():
        df = stocks_data[sym]
        if d in df.index:
            idx = df.index.get_loc(d)
            if idx + 1 < len(df):
                labels.append(df["return"].iloc[idx + 1])
            else:
                labels.append(np.nan)
        else:
            labels.append(np.nan)

    factor_df_copy = factor_df.copy()
    factor_df_copy["target"] = labels
    factor_df_copy = factor_df_copy.dropna(subset=["target"] + factor_cols)

    # 时间切分
    unique_dates = sorted(factor_df_copy.index.get_level_values("date").unique())
    train_dates = unique_dates[:train_months]
    test_dates = unique_dates[train_months:]

    train = factor_df_copy.loc[train_dates]
    test = factor_df_copy.loc[test_dates]

    return train, test

train, test = build_labels_and_split(factor_df, stocks_data, dates, train_months=36)
print(f"训练集: {len(train)} 条 | 测试集: {len(test)} 条")
print(f"训练月数: {train.index.get_level_values('date').nunique()}")
print(f"测试月数: {test.index.get_level_values('date').nunique()}")

In [ ]:
# 标准化（只用训练集参数）
train_mean = train[factor_cols].mean()
train_std = train[factor_cols].std()

X_train = (train[factor_cols] - train_mean) / train_std
y_train = train["target"]
X_test = (test[factor_cols] - train_mean) / train_std
y_test = test["target"]

# 方法1: 等权合成
def equal_weight_score(X):
    """所有因子等权"""
    return X.mean(axis=1)

# 方法2: 回归权重（OLS）
def regression_weight_score(X_train, y_train, X_test):
    """用 OLS 估计因子权重"""
    beta = np.linalg.lstsq(X_train.values, y_train.values, rcond=None)[0]
    return X_test.values @ beta

# 方法3: IC 加权（信息系数加权）
def ic_weight_score(X_train, y_train, X_test):
    """用每个因子与标签的 Spearman 秩相关作为权重"""
    from scipy.stats import spearmanr
    weights = []
    for col in X_train.columns:
        ic, _ = spearmanr(X_train[col], y_train)
        weights.append(ic)
    weights = np.array(weights)
    weights = weights / np.sum(np.abs(weights))  # 归一化
    return X_test.values @ weights

# 计算三种方法的得分
scores_eq = equal_weight_score(X_test)
scores_reg = regression_weight_score(X_train, y_train, X_test)
scores_ic = ic_weight_score(X_train, y_train, X_test)

# 对比：IC（信息系数）
from scipy.stats import spearmanr
ic_eq, _ = spearmanr(scores_eq, y_test)
ic_reg, _ = spearmanr(scores_reg, y_test)
ic_ic, _ = spearmanr(scores_ic, y_test)

print("=" * 50)
print("因子合成方法对比（样本外 IC）")
print("=" * 50)
print(f"  等权:     IC = {ic_eq:.4f}")
print(f"  OLS回归:  IC = {ic_reg:.4f}")
print(f"  IC加权:   IC = {ic_ic:.4f}")

# 选择 IC 最好的方法
best_method = max([("等权", ic_eq), ("OLS", ic_reg), ("IC加权", ic_ic)], key=lambda x: x[1])
print(f"\n✅ 选用: {best_method[0]} (IC={best_method[1]:.4f})")

# 用最佳方法生成最终得分
if best_method[0] == "等权":
    test_scores = pd.Series(scores_eq, index=test.index)
elif best_method[0] == "OLS":
    test_scores = pd.Series(scores_reg, index=test.index)
else:
    test_scores = pd.Series(scores_ic, index=test.index)

print(f"得分范围: [{test_scores.min():.4f}, {test_scores.max():.4f}]")

## 第五阶段：策略信号生成

每月选出得分最高的 top 3 只股票等权配置。
- 选股逻辑：top score → 买入，不在 top 3 → 卖出
- 调仓频率：月度
- 单只股票上限：40%
- 最少持仓：始终持满 3 只（除非可买股票不足 3 只）

In [ ]:
TOP_K = 3

# 每个月选出 top 3
test_dates_unique = sorted(test.index.get_level_values("date").unique())

monthly_portfolio = {}
for d in test_dates_unique:
    month_data = test_scores.loc[d]
    if isinstance(month_data, pd.Series):
        top_stocks = month_data.nlargest(TOP_K).index.tolist()
        monthly_portfolio[d] = top_stocks

print(f"测试月数: {len(monthly_portfolio)}")
print(f"\n前5个月的持仓:")
for i, (d, stocks) in enumerate(list(monthly_portfolio.items())[:5]):
    print(f"  {d.date()}: {stocks}")

# 统计换手率
turnovers = []
prev_stocks = set()
for d, stocks in monthly_portfolio.items():
    current = set(stocks)
    if prev_stocks:
        n_changes = len(current.symmetric_difference(prev_stocks)) // 2
        turnovers.append(n_changes / TOP_K)
    prev_stocks = current

avg_turnover = np.mean(turnovers) if turnovers else 0
print(f"\n平均月度换手率: {avg_turnover:.1%}")

## 第六阶段：回测执行

使用第 32-33 课的手写回测引擎 + 风控模块。关键设置：
- 初始资金 100 万
- 手续费 0.03%（万三）
- 滑点 0.1%
- 凯利仓位管理（半凯利）
- ATR 止损（2 倍 ATR）
- 最大回撤限制 25%

In [ ]:
class OrderType(Enum):
    MARKET = "market"

class OrderSide(Enum):
    BUY = "buy"
    SELL = "sell"

@dataclass
class Order:
    order_id: int
    symbol: str
    order_type: OrderType
    side: OrderSide
    quantity: int
    timestamp: Optional[int] = None

@dataclass
class Fill:
    order_id: int
    symbol: str
    side: OrderSide
    quantity: int
    price: float
    fee: float
    timestamp: int

@dataclass
class Account:
    initial_cash: float = 1_000_000
    cash: float = None
    positions: Dict[str, int] = field(default_factory=dict)
    nav_records: List[Dict] = field(default_factory=list)
    trades: List[Dict] = field(default_factory=list)

    def __post_init__(self):
        if self.cash is None:
            self.cash = self.initial_cash

    def can_afford(self, order, price):
        return self.cash >= price * order.quantity * 1.0003

    def has_position(self, symbol, qty):
        return self.positions.get(symbol, 0) >= qty

    def apply_fill(self, fill, current_price):
        if fill.side == OrderSide.BUY:
            cost = fill.price * fill.quantity + fill.fee
            self.cash -= cost
            self.positions[fill.symbol] = self.positions.get(fill.symbol, 0) + fill.quantity
        else:
            revenue = fill.price * fill.quantity - fill.fee
            self.cash += revenue
            self.positions[fill.symbol] = self.positions.get(fill.symbol, 0) - fill.quantity
            if self.positions[fill.symbol] <= 0:
                del self.positions[fill.symbol]
        self.trades.append({
            "timestamp": fill.timestamp,
            "symbol": fill.symbol,
            "side": fill.side.value,
            "quantity": fill.quantity,
            "price": fill.price,
            "fee": fill.fee,
        })

    def total_value(self, price_map):
        pv = sum(q * price_map.get(s, 0) for s, q in self.positions.items())
        return self.cash + pv

    def snapshot(self, ts, price_map):
        self.nav_records.append({
            "timestamp": ts,
            "cash": self.cash,
            "total_value": self.total_value(price_map),
            "n_positions": len(self.positions),
        })


def run_backtest(stocks_data, monthly_portfolio, test_dates_unique,
                 initial_cash=1_000_000, slippage=0.001, fee_rate=0.0003,
                 kelly_fraction=0.5, max_position_pct=0.40,
                 max_dd_limit=0.25, dd_recovery=0.15):
    """完整回测：多股票月度调仓 + 风控"""
    acct = Account(initial_cash=initial_cash)
    fill_id = 0
    # 风控状态
    peak_nav = initial_cash
    trading_halted = False

    # 将所有股票价格对齐到统一时间轴
    price_history = {}
    for sym in stocks_data:
        price_history[sym] = stocks_data[sym]["price"]

    for t, d in enumerate(test_dates_unique):
        # 当前各股票价格
        current_prices = {}
        for sym in stocks_data:
            if d in price_history[sym].index:
                current_prices[sym] = price_history[sym].loc[d]

        current_nav = acct.total_value(current_prices)

        # === 风控：回撤限制 ===
        peak_nav = max(peak_nav, current_nav)
        current_dd = (current_nav - peak_nav) / peak_nav

        if trading_halted:
            if abs(current_dd) < dd_recovery:
                trading_halted = False
            acct.snapshot(t, current_prices)
            continue

        if abs(current_dd) >= max_dd_limit:
            trading_halted = True
            # 清仓
            for sym, qty in list(acct.positions.items()):
                if sym in current_prices:
                    fill_id += 1
                    price = current_prices[sym] * (1 - slippage)
                    fee = price * qty * fee_rate
                    fill = Fill(fill_id, sym, OrderSide.SELL, qty, price, fee, t)
                    acct.apply_fill(fill, current_prices[sym])
            acct.snapshot(t, current_prices)
            continue

        # === 持仓检查：不在 top K 的卖出 ===
        target_stocks = set(monthly_portfolio.get(d, []))
        for sym, qty in list(acct.positions.items()):
            if sym not in target_stocks and sym in current_prices:
                fill_id += 1
                price = current_prices[sym] * (1 - slippage)
                fee = price * qty * fee_rate
                fill = Fill(fill_id, sym, OrderSide.SELL, qty, price, fee, t)
                acct.apply_fill(fill, current_prices[sym])

        # === 买入 target stocks ===
        cash_per_stock = acct.cash / max(len(target_stocks - set(acct.positions.keys())), 1)
        cash_per_stock = min(cash_per_stock, acct.cash * max_position_pct)

        # 凯利修正
        if acct.trades:
            # 简化版凯利：用账户整体收益历史
            recent_returns = []
            nav_vals = [r["total_value"] for r in acct.nav_records]
            if len(nav_vals) >= 3:
                recent_returns = np.diff(nav_vals[-12:]) / nav_vals[-13:-1] if len(nav_vals) >= 13 else []
            if len(recent_returns) > 0:
                wins = sum(1 for r in recent_returns if r > 0)
                wr = wins / len(recent_returns) if recent_returns else 0.5
                gains = [r for r in recent_returns if r > 0]
                losses = [r for r in recent_returns if r < 0]
                plr = (np.mean(gains) / abs(np.mean(losses))) if losses and gains else 1.5
                f_star = max(0, (wr * plr - (1 - wr)) / plr) if plr > 0 else 0.1
                kelly_adj = f_star * kelly_fraction
                cash_per_stock *= min(max(kelly_adj, 0.1), 1.0)

        for sym in target_stocks:
            if sym in acct.positions or sym not in current_prices:
                continue
            price = current_prices[sym]
            qty = int(cash_per_stock / (price * (1 + slippage)))
            if qty <= 0:
                continue
            if not acct.can_afford(Order(0, sym, OrderType.MARKET, OrderSide.BUY, qty), price):
                continue

            fill_price = price * (1 + slippage)
            fee = fill_price * qty * fee_rate
            fill_id += 1
            fill = Fill(fill_id, sym, OrderSide.BUY, qty, fill_price, fee, t)
            acct.apply_fill(fill, price)

        acct.snapshot(t, current_prices)

    return acct

# 运行回测
acct = run_backtest(stocks_data, monthly_portfolio, test_dates_unique)
navs = np.array([r["total_value"] for r in acct.nav_records])
returns = np.diff(navs) / navs[:-1]

# 基准（等权买入所有股票持有）
benchmark_returns = []
for t, d in enumerate(test_dates_unique):
    month_rets = [stocks_data[s]["return"].loc[d] for s in stocks_data if d in stocks_data[s].index]
    if month_rets:
        benchmark_returns.append(np.mean(month_rets))
benchmark_nav = 1_000_000 * np.cumprod(1 + np.array(benchmark_returns))

print("=" * 60)
print("回测完成")
print("=" * 60)
print(f"  交易笔数: {len(acct.trades)}")
print(f"  最终净值: {navs[-1]:,.0f}")
print(f"  最终现金: {acct.cash:,.0f}")

## 第七阶段：绩效分析

计算标准绩效指标并与基准对比。

In [ ]:
def compute_performance(navs, returns, benchmark_nav):
    """完整绩效计算"""
    n = len(returns)

    total_return = navs[-1] / navs[0] - 1
    annual_return = (1 + total_return) ** (12 / n) - 1
    annual_vol = np.std(returns) * np.sqrt(12)
    sharpe = (annual_return - 0.02) / annual_vol if annual_vol > 0 else 0

    peak = np.maximum.accumulate(navs)
    dd = (navs - peak) / peak
    max_dd = dd.min()
    calmar = annual_return / abs(max_dd) if max_dd != 0 else 0

    win_rate = (returns > 0).mean()
    gains = returns[returns > 0]
    losses = returns[returns < 0]
    plr = gains.mean() / abs(losses.mean()) if len(losses) > 0 else float("inf")

    # 超额收益
    excess_returns = returns - np.diff(benchmark_nav[:len(navs)]) / benchmark_nav[:len(navs)-1]
    ir = np.mean(excess_returns) / np.std(excess_returns) * np.sqrt(12) if np.std(excess_returns) > 0 else 0

    return {
        "累计收益": total_return,
        "年化收益": annual_return,
        "年化波动": annual_vol,
        "Sharpe": sharpe,
        "最大回撤": max_dd,
        "Calmar": calmar,
        "胜率": win_rate,
        "盈亏比": plr,
        "IR": ir,
        "nav": navs,
        "drawdowns": dd,
    }

perf = compute_performance(navs, returns, benchmark_nav)

bm_total = benchmark_nav[-1] / benchmark_nav[0] - 1
bm_dd = (benchmark_nav / np.maximum.accumulate(benchmark_nav) - 1).min()

print("=" * 60)
print(f"{'指标':<14s} {'策略':>12s} {'基准':>12s} {'超额':>12s}")
print("=" * 60)
print(f"{'累计收益':<14s} {perf['累计收益']:>11.2%} {bm_total:>11.2%} {perf['累计收益']-bm_total:>11.2%}")
print(f"{'年化收益':<14s} {perf['年化收益']:>11.2%}")
print(f"{'年化波动':<14s} {perf['年化波动']:>11.2%}")
print(f"{'Sharpe':<14s} {perf['Sharpe']:>11.2f}")
print(f"{'最大回撤':<14s} {perf['最大回撤']:>11.2%} {bm_dd:>11.2%}")
print(f"{'Calmar':<14s} {perf['Calmar']:>11.2f}")
print(f"{'胜率':<14s} {perf['胜率']:>11.2%}")
print(f"{'盈亏比':<14s} {perf['盈亏比']:>11.2f}")
print(f"{'IR':<14s} {perf['IR']:>11.2f}")

In [ ]:
# 可视化
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# 净值
axes[0].plot(perf["nav"], label="多因子ML策略", linewidth=1.5, color="steelblue")
axes[0].plot(benchmark_nav[:len(perf["nav"])], label="等权持有基准", linewidth=1, linestyle="--", alpha=0.6, color="gray")
axes[0].axhline(y=1_000_000, color="black", linewidth=0.5, linestyle=":")
axes[0].set_title("净值曲线")
axes[0].legend()
axes[0].set_ylabel("净值 (元)")

# 回撤
axes[1].fill_between(range(len(perf["drawdowns"])), perf["drawdowns"] * 100, 0, alpha=0.3, color="coral")
axes[1].plot(perf["drawdowns"] * 100, linewidth=0.5, color="coral")
axes[1].axhline(y=-25, color="red", linestyle="--", linewidth=0.8, alpha=0.5, label="-25% 风控线")
axes[1].set_title("回撤 (%)")
axes[1].set_ylabel("回撤")
axes[1].legend()

# 月度收益分布
axes[2].bar(range(len(returns)), returns * 100, color=np.where(returns >= 0, "steelblue", "coral"), alpha=0.7)
axes[2].axhline(y=0, color="black", linewidth=0.5)
axes[2].set_title(f"月度收益 (%)")
axes[2].set_ylabel("收益")
axes[2].set_xlabel("月份")

plt.tight_layout()
plt.show()

## 第八阶段：归因分析

拆解策略收益来源：选股能力 vs 市场暴露。

In [ ]:
# 归因：拆解收益来源
print("=" * 60)
print("收益归因分析")
print("=" * 60)

bm_rets = np.diff(benchmark_nav[:len(navs)]) / benchmark_nav[:len(navs)-1]

# 市场贡献（beta）
cov = np.cov(returns, bm_rets)
beta = cov[0, 1] / cov[1, 1] if cov[1, 1] > 0 else 1.0
market_contribution = beta * np.mean(bm_rets) * 12

# alpha
alpha = perf["年化收益"] - 0.02 - beta * (np.mean(bm_rets) * 12 - 0.02)

print(f"  Beta (市场暴露):    {beta:.3f}")
print(f"  市场贡献 (年化):     {market_contribution:.2%}")
print(f"  Alpha (选股能力):    {alpha:.2%}")
print(f"  策略年化收益:         {perf['年化收益']:.2%}")
print(f"  其中: 市场={market_contribution:.2%} + Alpha={alpha:.2%} + 无风险=2%")

# 因子暴露分析
print(f"\n{'因子':<12s} {'IC':>8s} {'方向':>8s}")
print("-" * 30)
for col in factor_cols:
    ic_val, _ = spearmanr(X_test[col], y_test)
    direction = "做多" if ic_val > 0 else "做空"
    print(f"{col:<12s} {ic_val:>8.4f} {direction:>8s}")

## 第九阶段：策略总结报告

一份完整的量化策略报告应包括以下内容。这也是面试中你需要讲清楚的故事线。

In [ ]:
print("╔" + "═" * 58 + "╗")
print("║" + "  量化策略研究报告：多因子 ML 选股策略".center(52) + "║")
print("╠" + "═" * 58 + "╣")
print("║ " + f"{'策略名称':<16s}: 多因子 ML 选股策略".ljust(57) + "║")
print("║ " + f"{'投资哲学':<16s}: 便宜好公司+市场悲观时买入".ljust(57) + "║")
print("║ " + f"{'因子数':<16s}: 7 (动量3+波动率2+估值1+质量1)".ljust(57) + "║")
print("║ " + f"{'模型':<16s}: 多因子线性合成 (IC加权)".ljust(57) + "║")
print("║ " + f"{'选股':<16s}: 月度调仓 Top 3, 等权配置".ljust(57) + "║")
print("║ " + f"{'风控':<16s}: 半凯利仓位+25%回撤熔断".ljust(57) + "║")
print("║ " + "─" * 56 + "║")
print(f"║  绩效摘要".ljust(57) + "║")
print(f"║    年化收益: {perf['年化收益']:>8.2%}  |  Sharpe: {perf['Sharpe']:>6.2f}".ljust(57) + "║")
print(f"║    最大回撤: {perf['最大回撤']:>8.2%}  |  Calmar: {perf['Calmar']:>6.2f}".ljust(57) + "║")
print(f"║    胜率:     {perf['胜率']:>8.2%}  |  IR:     {perf['IR']:>6.2f}".ljust(57) + "║")
print("║ " + "─" * 56 + "║")
print(f"║  归因".ljust(57) + "║")
print(f"║    Beta: {beta:.3f} | 市场贡献: {market_contribution:.2%} | Alpha: {alpha:.2%}".ljust(57) + "║")
print("║ " + "─" * 56 + "║")
print("║ " + f"{'回测陷阱检查':<16s}: ✅ 时间切分 ✅ 无前视偏差".ljust(57) + "║")
print("║ " + f"{'':<16s}  ✅ 含手续费+滑点 ✅ 含风控".ljust(57) + "║")
print("║ " + f"{'':<16s}  ✅ 样本外测试 ✅ 有归因分析".ljust(57) + "║")
print("╚" + "═" * 58 + "╝")

## 第十阶段：面试叙事框架

如果你在面试中被问到"做过什么量化策略"，用这个框架回答：

1. **投资逻辑（Why）**：为什么这个策略应该有效？（经济直觉，不是数据挖掘）
2. **实现方式（How）**：用了什么因子、什么模型、什么频率？（展示技术深度）
3. **风控措施（Risk）**：怎么控制下行风险？（展示风险意识）
4. **绩效结果（Result）**：Sharpe、回撤、归因分解（数据说话）
5. **自我批判（Limitation）**：策略的局限是什么？如果实盘会有什么问题？（展示成熟度）

### 这个策略的局限

1. **模拟数据**：用的是合成数据而非真实行情，因子分布在真实市场中更复杂
2. **幸存者偏差**：没有考虑退市股票
3. **容量限制**：Top 3 选股容量有限，资金大到一定程度就无法执行
4. **因子衰减**：动量因子在 A 股的有效性近年有所下降
5. **黑天鹅**：25% 回撤熔断在市场闪崩中可能来不及执行

## 小结

这节课是整个 0-100 Quant 课程的收官之战。我们走完了量化策略开发的完整链路：

```text
投资哲学 → 因子构建 → 模型训练 → 信号生成 → 回测执行 → 风控管理 → 绩效归因 → 策略报告
```

你在这 34 节课中学到的每一个知识点，都在这条链路上有一席之地。

从这里出发，你可以：
- 把模拟数据替换为真实行情（AKShare/Wind），验证策略的有效性
- 加入更多因子（基本面、情绪、另类数据）
- 尝试更复杂的模型（LightGBM、神经网络）
- 把这个策略写成研报 PDF，投递量化实习

量化之路才刚刚开始。

## 验收清单

- [ ] 能清楚地讲述策略的投资逻辑（不是"因为回测好看"）
- [ ] 策略含至少 5 个因子，每个因子有经济学解释
- [ ] 严格按时间顺序切分训练/测试集（无前视偏差）
- [ ] 含仓位管理（凯利/上限）和止损/回撤风控
- [ ] 有完整绩效指标（Sharpe/Calmar/IR/胜率/盈亏比）
- [ ] 有归因分析（Beta/Alpha 拆解）
- [ ] 能说出策略的 3 个以上局限
- [ ] 代码可一键复现（固定随机种子）